In [ ]:
# ============================================================
# CELL 1 — Install + Clone
# ============================================================
import os, sys, subprocess, time
from pathlib import Path

# ---------- CONFIG ----------
REPO_URL          = "https://github.com/Imsachin010/salespath_env.git"
MODEL_NAME        = "unsloth/Qwen2.5-7B-Instruct"   # 7B Model for final submission
ENV_URL           = "http://127.0.0.1:8000"
OUTPUT_DIR        = "/content/salespath_out"
# -----------------------------------------

def run(cmd, check=True, cwd=None):
    print(f"\n$ {cmd}")
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, cwd=cwd)
    if r.stdout: print(r.stdout.strip())
    if r.stderr: print(r.stderr.strip())
    if check and r.returncode != 0:
        raise RuntimeError(f"Command failed ({r.returncode}): {cmd}")
    return r

!nvidia-smi
print("Python:", sys.version)

# Install dependencies
!pip install -q -U pip
!pip uninstall -y openenv 2>/dev/null || true
!pip install -q fastapi uvicorn pydantic httpx openenv-core torch transformers trl unsloth datasets pyarrow huggingface_hub matplotlib

# Clone repo
if not Path("/content/salespath_env").exists():
    run(f"git clone {REPO_URL} /content/salespath_env")
else:
    print("Repo already cloned.")

REPO_ROOT = "/content/salespath_env"
os.chdir(REPO_ROOT)
print("Working dir:", os.getcwd())

# Install package in editable mode
run("pip install -q -e .")
run("python -c \"import salespath_env; print('salespath_env import OK')\"")
run("python -c \"import openenv.core; print('openenv.core import OK')\"")

# HF Login
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    from huggingface_hub import login
    login(token=hf_token)
    print("HF login OK")
else:
    print("HF_TOKEN not set.")

print("\n✅ Setup complete.")

In [ ]:
# Pull the fix we just pushed
!git pull origin main

In [ ]:

!PYTORCH_ALLOC_CONF=expandable_segments:True \
python -m training.grpo_train \
    --mode grpo \
    --model-name unsloth/Qwen2.5-7B-Instruct \
    --grpo-steps 150 \
    --grpo-dataset-size 128 \
    --num-generations 2 \
    --max-completion-length 128 \
    --per-device-train-batch-size 2 \
    --gradient-accumulation-steps 8 \
    --output-dir /content/salespath_out \
    --logging-steps 10
